In [36]:
import numpy as np
import time
import pandas as pd
import MultiSuSiE
import argparse
from IPython.display import Markdown as md
import os

# Change the current working directory to the desired path
os.chdir("/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/")
print("Current working directory:", os.getcwd())


# Assign the input values to variables
num_causal = 3
LD_BLOCK = 2
h2_num = 2

Current working directory: /scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb


In [32]:
wrk_dir = f"/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_50/causal_num_{num_causal}/"
data_dir = os.path.join(wrk_dir, "summary_data/")
# Reading the data (equivalent to read.table in R, using numpy's genfromtxt)
zfile = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}",
                      delimiter=' ', names=True, dtype=None, encoding='utf-8')
zscore_1 = zfile['zscore_1']
zscore_2 = zfile['zscore_2']
N_1 = zfile['N_1']
N_2 = zfile['N_2']
# Create z_list and N_list
z_list = [zscore_1, zscore_2]
N_list = [300000,300000]
# Reading the covariance matrices using numpy
EU_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD1", delimiter=' ')
BB_cov = np.genfromtxt(f"{data_dir}CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{int(h2_num)}.LD2", delimiter=' ')

# Create R_mat_list as a dictionary of numpy arrays
R_list = [EU_cov,  BB_cov]
# summary_stat_sd_list and R_mat_list follow the R input structure
#maf_EU = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_ld_mafocus/risk_loci_ld_eur/maf_loci_{LD_BLOCK}.txt')
#maf_BB = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_ld_mafocus/risk_loci_ld_afr/maf_loci_{LD_BLOCK}.txt')
#maf_list = [maf_EU, maf_BB]

In [33]:
maf_EU = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_eur/maf_loci_{LD_BLOCK}_maf.txt')
maf_BB = np.loadtxt(f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/risk_loci_ld_afr/maf_loci_{LD_BLOCK}_maf.txt')
maf_list = [maf_EU, maf_BB]

In [34]:
# Record the start time
start_time = time.time()

ss_fit = MultiSuSiE.multisusie_rss(

    z_list = z_list,

    R_list = R_list,

    rho = np.array([[1, 0.8], [0.8, 1]]),

    population_sizes = N_list,

    L = 10,

    scaled_prior_variance = 0.2,

    low_memory_mode = False,
    
    min_abs_corr = 0.5,
    
    single_population_mac_thresh = 20,
    
    maf_list = maf_list,
    
    coverage = 0.95

)

# Record the end time
end_time = time.time()

# Calculate the time taken
time_taken = (end_time - start_time)/60

#indices = np.where(zfile['Signal'] != 0)[0]

#print(indices)


#print(ss_fit.sets[0])
#print(ss_fit.sets[3])
file_path = f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_50/multi_susie_result/MultiSuSiE_CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}_output'

# Save the run time to a text file
runtime_file = f'{file_path}_runtime.txt'
with open(runtime_file, 'w') as f:
    f.write(f"Time taken: {time_taken:.8f} minutes\n")

Censored 0 variants in population 0 due to low population-specific MAC
Censored 0 variants in population 1 due to low population-specific MAC


In [35]:
time_taken

0.009418789545694988

In [18]:
# Convert structured array to DataFrame
df_zfile = pd.DataFrame(zfile)

# Add the ss_fit.pip values as a new column in the DataFrame
df_zfile['pip'] = ss_fit.pip

In [13]:
print(ss_fit.sets[0])
print(ss_fit.sets[3])

[array([378, 386, 388, 389, 391, 397, 417]), array([794, 798, 805, 813]), array([   0,    1,    2, ..., 1806, 1807, 1808]), array([   0,    1,    2, ..., 1806, 1807, 1808]), array([   0,    1,    2, ..., 1806, 1807, 1808]), array([   0,    1,    2, ..., 1806, 1807, 1808]), array([   0,    1,    2, ..., 1806, 1807, 1808]), array([   0,    1,    2, ..., 1806, 1807, 1808]), array([   0,    1,    2, ..., 1806, 1807, 1808]), []]
[ True  True False False False False False False False False]


In [12]:
filtered_sets = [ss_fit.sets[0][i] for i in range(len(ss_fit.sets[0])) if ss_fit.sets[3][i] == True]
filtered_sets

[array([378, 386, 388, 389, 391, 397, 417]), array([794, 798, 805, 813])]

In [14]:
help(MultiSuSiE.multisusie_rss)

Help on function multisusie_rss in module MultiSuSiE.susiepy_ss:

multisusie_rss(R_list, population_sizes, b_list=None, s_list=None, z_list=None, varY_list=None, rho=0.75, L=10, scaled_prior_variance=0.2, prior_weights=None, standardize=False, pop_spec_standardization=True, estimate_residual_variance=True, estimate_prior_variance=True, estimate_prior_method='early_EM', pop_spec_effect_priors=True, iter_before_zeroing_effects=5, prior_tol=1e-09, max_iter=100, tol=0.001, verbose=False, coverage=0.95, min_abs_corr=0, float_type=<class 'numpy.float32'>, low_memory_mode=False, recover_R=False, single_population_mac_thresh=20, mac_list=None, multi_population_maf_thresh=0, maf_list=None)
    Top-level function for running MultiSuSiE
    
    This function takes takes standard GWAS summary statistics, converts
    them to sufficient statistics, and runs MultiSuSiE on them.
    
    Parameters
    ----------
    multisusie_rss accepts two combinations of input parameters:
    1. b_list, s_list,

In [10]:
# Create a new empty DataFrame to store the results
filtered_df = pd.DataFrame()

# For each filtered set, find corresponding SNPs by their index in zfile and label with CS index
for idx, cs_set in enumerate(filtered_sets):
    for snp_index in cs_set:
        # Select the row in df_zfile based on the SNP index (rather than by POS)
        if snp_index < len(df_zfile):
            matching_snp_df = df_zfile.iloc[[snp_index]].copy()
            # Add the CS index to the matched SNP
            matching_snp_df['CS'] = idx + 1
            # Append the matching SNP data to the new filtered DataFrame
            filtered_df = pd.concat([filtered_df, matching_snp_df])

In [28]:

# Define file names for txt files
zfile_output = f'{file_path}_snp.txt'
filtered_output = f'{file_path}_cs.txt'

# Save the df_zfile DataFrame to a .txt file with tab separation
df_zfile.to_csv(zfile_output, sep='\t', index=False)

# Save the filtered_df DataFrame to a .txt file with tab separation
filtered_df.to_csv(filtered_output, sep='\t', index=False)

In [ ]:
# Use pickle to save the MultiSuSiE output directly

import pickle


file_path = f'/scratch/negishi/chen4422/hapnest/Simulation_update/simu_region_1mb/shared_50/multi_susie_result/MultiSuSiE_CAUSAL_{num_causal}_LOCI_{LD_BLOCK}_h2_{h2_num}_output'

# Open the file in binary mode
with open(file_path, 'wb') as file:
    # Serialize and write the variable to the file
    pickle.dump(ss_fit, file)
    
    
with open(file_path, 'rb') as file:
    # Deserialize and retrieve the variable from the file
    loaded_data = pickle.load(file)